<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Zeshengliu213/ai-for-materials/blob/main/09%20-%20Generative%20Models/09a_vae_selfies.ipynb)

# 09a · 分子的连续坐标：SELFIES 和变分自编码器

> 对应 ai4chem `05 - Generative Models` 的 VAE 部分。原课程加载一个预训练的 VAE 只做可视化；这里我们自己训练一个（CPU 上约 4 分钟），并且用 **SELFIES** 代替 SMILES，顺便检验一个常见的说法："SELFIES 保证 100% 有效"。

08a 的 LSTM 一个字一个字地写分子，中间没有任何地方存在"这个分子的向量表示"。这一节的模型不一样：它给每个分子一个**连续的坐标**，于是我们可以在分子之间插值、在化学空间里"走路"。

1. SELFIES：一种永远合法的分子字符串
2. 自编码器和变分自编码器
3. 训练一个分子 VAE
4. 重构：它记住分子了吗？
5. 潜空间：插值和 PCE 的分布
6. 从先验采样：有效 ≠ 合理
7. RNN 还是 VAE？

### 这个 notebook 怎么用

| 标记 | 意思 | 你要做什么 |
|---|---|---|
| 📘 | 讲解 | 读一读 |
| 🤔 | 先猜一猜 | **运行之前**先想想会输出什么 |
| ▶️ | 运行看看 | 选中代码格，按 `Shift + Enter` |
| ✏️ | 改一改 | 改代码里标了 `# ✏️` 的地方，再运行 |
| 🏋️ | 练习 | 自己写代码；参考答案在 `solutions/` 文件夹 |
| 💡 | 小贴士 | 有用但不是必须记住的东西 |

In [ ]:
# colab-setup (本地运行时自动跳过)
import os, sys

if "google.colab" in sys.modules:
    _repo_dir = "/content/ai-for-materials"
    if not os.path.exists(_repo_dir):
        os.system("git clone --depth 1 https://github.com/Zeshengliu213/ai-for-materials.git " + _repo_dir)
    os.chdir(os.path.join(_repo_dir, "09 - Generative Models"))
    get_ipython().run_line_magic("pip", "install -q rdkit selfies")


In [ ]:
try:
    import selfies
except ImportError:
    %pip install -q selfies rdkit
    import selfies

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import selfies as sf
from torch import nn
from rdkit import Chem, RDLogger
from rdkit.Chem import Draw, Descriptors

RDLogger.DisableLog("rdApp.*")
torch.manual_seed(0)
np.random.seed(0)
print("selfies 版本:", sf.__version__)

<!-- ai4m-style -->
> **📘 课程统一绘图风格**：下面这格加载课程自带的 `ai4m.mplstyle`（配色与课程网站同源，无上右边框，仅横向浅网格，保存默认 300 dpi）。`C` 是课程配色，手动指定颜色时写 `color=C.crimson`。


In [ ]:
# ai4m-style：课程统一绘图风格（本地与 Colab 通用）
import pathlib, sys

sys.path.insert(0, str(pathlib.Path.cwd().parent))   # course/ 根目录
import ai4m_style as ai4m

C = ai4m.use()        # 应用 ai4m.mplstyle，返回课程配色


In [ ]:
import os
import pandas as pd

DATA_DIR = "data"
if not os.path.exists(os.path.join(DATA_DIR, "cep.csv")):
    DATA_DIR = "."
if not os.path.exists(os.path.join(DATA_DIR, "cep.csv")):
    try:
        from google.colab import files
        print("请上传 cep.csv")
        files.upload()
    except ImportError:
        raise FileNotFoundError("找不到 cep.csv，请把它放在 data/ 文件夹或和 notebook 同一个文件夹里")

df = pd.read_csv(os.path.join(DATA_DIR, "cep.csv"))
print("读取成功！一共", len(df), "个分子")

---
## 1. SELFIES

📘 08a 里 LSTM 生成的 SMILES 仍有百分之几是**无效**的：括号没配对、环号落单、化合价不对。这是用 SMILES 做生成模型的老问题。

**SELFIES**（SELF-referencIng Embedded Strings，Krenn 等人 2020）换了一套语法，让"任何 token 序列都对应一个化学上合法的分子"。它的做法是：

- 每个 token 自带"接下来要连几根键"的信息（`[=C]`、`[#N]`……）；
- 成环和支链用**相对计数**表示（`[Ring1]`、`[Branch1]` 后面跟一个表示长度的 token），而不是需要配对的数字；
- 解码时如果某个指令会违反化合价，就**自动截断或修正**。

换句话说：SMILES 的语法允许你写出错的东西，SELFIES 的语法不允许。

In [ ]:
examples = ["c1ccsc1", "c1ccc2c(c1)sc1c3ccccc3sc21", df["smiles"][0]]
for s in examples:
    encoded = sf.encoder(s)
    print("SMILES :", s)
    print("SELFIES:", encoded)
    print("解码回来:", sf.decoder(encoded))
    print("和原分子相同?", Chem.CanonSmiles(sf.decoder(encoded)) == Chem.CanonSmiles(s), "\n")

📘 SELFIES 串长得很奇怪，但它和 SMILES 一一对应、可以来回转换。

🤔 SELFIES"永远有效"是什么意思？做个实验：把一个合法字符串里的 token **随机改掉一个**，SMILES 和 SELFIES 分别会怎样？

In [ ]:
import random
random.seed(0)

smiles = "c1ccc2c(c1)sc1c3ccccc3sc21"                 # BTBT
selfies_str = sf.encoder(smiles)
sf_tokens = list(sf.split_selfies(selfies_str))
alphabet_demo = sorted(sf.get_semantic_robust_alphabet())

n_trials = 200
smiles_ok = selfies_ok = 0
for _ in range(n_trials):
    chars = list(smiles)
    chars[random.randrange(len(chars))] = random.choice("cCnNoOsS123()=")
    smiles_ok += Chem.MolFromSmiles("".join(chars)) is not None

    toks = list(sf_tokens)
    toks[random.randrange(len(toks))] = random.choice(alphabet_demo)
    decoded = sf.decoder("".join(toks))
    selfies_ok += (Chem.MolFromSmiles(decoded) is not None) if decoded else False

print(f"随机改一个 token 后还有效：SMILES {smiles_ok / n_trials:.1%}，SELFIES {selfies_ok / n_trials:.1%}")

📘 结果很干脆：随机改一个字符，SMILES 只有不到 20% 还能解析；SELFIES **100% 仍然是合法分子**。这就是它的卖点，对生成模型来说意味着采样出来的东西几乎总能解码成分子。

⚠️ 但"合法"不等于"合理"。第 6 节我们会看到这个区别有多重要。

▶️ 把整个数据集转成 SELFIES，建立字母表：

In [ ]:
selfies_list = [sf.encoder(s) for s in df["smiles"]]
token_lists = [list(sf.split_selfies(s)) for s in selfies_list]

alphabet = ["[nop]"] + sorted({t for ts in token_lists for t in ts})    # [nop] 是补齐符
stoi = {t: i for i, t in enumerate(alphabet)}
max_len = max(len(ts) for ts in token_lists)
print("字母表大小:", len(alphabet), " 最长:", max_len, " 平均长度:", np.mean([len(ts) for ts in token_lists]).round(1))
print(alphabet)

---
## 2. 自编码器和变分自编码器

### 2.1 自编码器

📘 **自编码器**（autoencoder）由两个网络组成：

```
分子 --[编码器]--> 低维向量 z --[解码器]--> 分子
```

训练目标很简单：让输出尽量等于输入。中间那个低维向量 $z$ 叫**潜向量**（latent vector），它被迫压缩了分子的全部信息。

这和 03a 的 PCA 很像！PCA 也是"降维再重建"，区别是 PCA 只能做线性变换，自编码器的编码器和解码器都是神经网络，可以非线性。

### 2.2 为什么要"变分"

🤔 普通自编码器训练好以后，随便拿一个向量 $z$ 交给解码器，能得到合理的分子吗？

📘 通常不能。普通自编码器只保证"训练过的那些点"能重建，点和点之间的空隙是**没有定义**的，解码出来往往是垃圾。

**变分自编码器**（VAE）加了两个改动来把潜空间"填满"：

1. **编码器输出的不是一个点，而是一个高斯分布**：均值 $\mu$ 和对数方差 $\log \sigma^2$。训练时从这个分布里随机采一个 $z$。于是同一个分子每次的 $z$ 都略有不同，解码器被迫让 $\mu$ **附近**的点也能解出这个分子；
2. **KL 散度惩罚**：逼着所有分子的 $(\mu, \sigma)$ 向标准正态分布 $N(0, I)$ 靠拢，让它们挤在一起、不要各占一方。

损失函数因此有两项：

$$\mathcal{L} = \underbrace{\text{重构误差}}_{\text{解码得准}} + \beta \cdot \underbrace{D_{KL}\big(q(z|x) \,\|\, N(0,I)\big)}_{\text{潜空间规整}}$$

$\beta$ 控制两者的平衡：$\beta$ 太大，潜空间很规整但什么都重构不出来（**后验坍缩**）；$\beta$ 太小，就退化成普通自编码器。

💡 **重参数化技巧**：直接"从分布里采样"这个操作没法求导。写成 $z = \mu + \sigma \cdot \epsilon$（其中 $\epsilon \sim N(0,I)$）之后，随机性被隔离在 $\epsilon$ 里，$\mu$ 和 $\sigma$ 就可以正常反向传播了。这是 VAE 能训练起来的关键。

---
## 3. 训练一个分子 VAE

📘 输入表示：每个分子是 `最长长度 × 字母表大小` 的 one-hot 矩阵，拉平成一个长向量。编码器和解码器都用简单的多层感知机（07a 学过）。

In [ ]:
indices = torch.tensor([[stoi[t] for t in ts] + [0] * (max_len - len(ts)) for ts in token_lists])
X = nn.functional.one_hot(indices, len(alphabet)).float().reshape(len(indices), -1)
print("输入矩阵:", tuple(X.shape), f"（{max_len} 个位置 × {len(alphabet)} 种 token）")

perm = np.random.RandomState(0).permutation(len(X))
test_idx, train_idx = perm[:3000], perm[3000:]
print("训练:", len(train_idx), " 测试:", len(test_idx))

In [ ]:
LATENT_DIM = 128                                   # ✏️ 潜空间维数
INPUT_DIM = X.shape[1]

class MolecularVAE(nn.Module):
    def __init__(self, input_dim, latent_dim):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(input_dim, 512), nn.ReLU(), nn.Linear(512, 256), nn.ReLU())
        self.fc_mu = nn.Linear(256, latent_dim)          # 分布的均值
        self.fc_logvar = nn.Linear(256, latent_dim)      # 分布的对数方差
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 256), nn.ReLU(),
            nn.Linear(256, 512), nn.ReLU(),
            nn.Linear(512, input_dim),
        )

    def encode(self, x):
        h = self.encoder(x)
        return self.fc_mu(h), self.fc_logvar(h)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = mu + torch.randn_like(mu) * torch.exp(0.5 * logvar)      # 重参数化技巧
        return self.decoder(z), mu, logvar

vae = MolecularVAE(INPUT_DIM, LATENT_DIM)
print("参数量:", sum(p.numel() for p in vae.parameters()))

In [ ]:
def decode_latent(z):
    """把潜向量解码成 SMILES 字符串。"""
    with torch.no_grad():
        logits = vae.decoder(z)
    token_ids = logits.reshape(len(z), max_len, len(alphabet)).argmax(dim=-1)
    out = []
    for row in token_ids.tolist():
        s = "".join(alphabet[i] for i in row if i != 0)     # 跳过 [nop]
        out.append(sf.decoder(s))
    return out

def canonical(smiles):
    """能解析就返回规范 SMILES，否则 None。"""
    mol = Chem.MolFromSmiles(smiles) if smiles else None
    return Chem.MolToSmiles(mol) if mol is not None and mol.GetNumAtoms() > 0 else None

▶️ 训练 40 轮，CPU 上大约 4 分钟。注意看两项损失的变化：

In [ ]:
BETA = 0.05                                        # ✏️ KL 项的权重
optimizer = torch.optim.Adam(vae.parameters(), lr=1e-3)
history = []

for epoch in range(40):
    order = np.random.permutation(train_idx)
    rec_total = kl_total = 0.0
    beta = BETA * min(1.0, (epoch + 1) / 13)       # 前 13 轮逐渐加大 beta（KL warm-up）
    for i in range(0, len(order), 256):
        xb = X[order[i:i + 256]]
        out, mu, logvar = vae(xb)
        rec = nn.functional.cross_entropy(
            out.reshape(-1, len(alphabet)), xb.reshape(-1, len(alphabet)).argmax(-1), reduction="sum") / len(xb)
        kl = -0.5 * torch.sum(1 + logvar - mu ** 2 - logvar.exp()) / len(xb)
        loss = rec + beta * kl
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        rec_total += rec.item()
        kl_total += kl.item()
    n_batches = len(order) / 256
    history.append({"epoch": epoch, "reconstruction": rec_total / n_batches, "KL": kl_total / n_batches})
    if epoch % 5 == 4 or epoch == 0:
        print(f"epoch {epoch:2d}: 重构 {rec_total / n_batches:6.2f}   KL {kl_total / n_batches:6.2f}")

vae.eval();

In [ ]:
hist = pd.DataFrame(history).set_index("epoch")
fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
hist["reconstruction"].plot(ax=axes[0], title="reconstruction loss")
hist["KL"].plot(ax=axes[1], title="KL divergence", color="tab:red")
for ax in axes:
    ax.set_xlabel("epoch")
plt.tight_layout(); plt.show()

📘 重构损失一路下降；KL 先升后缓慢下降——前期模型在"占地盘"（每个分子抢一块自己的区域），后期在 KL 的压力下慢慢收敛。

---
## 4. 重构：它记住分子了吗？

📘 最直接的检验：把测试集的分子编码成 $\mu$，再解码回来，看有多少能**一模一样**地还原。

In [ ]:
with torch.no_grad():
    mu_test, _ = vae.encode(X[test_idx])

reconstructed = decode_latent(mu_test[:1000])
originals = [canonical(s) for s in df["smiles"].values[test_idx[:1000]]]
recon_canonical = [canonical(s) for s in reconstructed]

exact = np.mean([a is not None and a == b for a, b in zip(recon_canonical, originals)])
valid = np.mean([a is not None for a in recon_canonical])
print(f"重构结果有效率: {valid:.3f}")
print(f"完全还原（规范 SMILES 相同）: {exact:.3f}")

🤔 完全还原率只有百分之十几。这个 VAE"失败"了吗？

📘 没有失败，但确实**不够好**。看看错在哪里：

In [ ]:
pd.DataFrame([{"original": o, "reconstructed": r, "same": o == r}
              for o, r in zip(originals[:6], recon_canonical[:6])])

In [ ]:
pairs = [(Chem.MolFromSmiles(o), Chem.MolFromSmiles(r)) for o, r in zip(originals, recon_canonical)
         if o and r and o != r][:4]
Draw.MolsToGridImage([m for pair in pairs for m in pair], molsPerRow=2, subImgSize=(280, 200),
                     legends=[x for _ in range(len(pairs)) for x in ("original", "reconstructed")])

📘 完全还原率只有 18% 左右，但**有效率高达 99%**：解码出来的几乎都是分子，只是不完全是原来那个。仔细看上面的表格和图，典型的错误是**骨架对了、细节错了**：噻吩变成呋喃（`cscc` → `cocc`）、少一个环、杂原子挪了位置。

原因是我们的解码器很简陋：一个 MLP 一次性"喷"出所有位置的 token，各个位置之间没有协调。真正的分子 VAE（Gómez-Bombarelli 等人 2018）用的是 **RNN 解码器**，一个 token 一个 token 地生成，可以参考已经生成的部分，重构率高得多。

💡 这里用 MLP 是为了让代码短、训练快（4 分钟 vs 半小时）。**知道一个模型为什么不够好，和知道怎么让它变好同样重要。**

---
## 5. 潜空间

### 5.1 在两个分子之间插值

📘 这是 VAE 最直观的能力：取两个分子的潜向量 $z_A$、$z_B$，沿直线取中间的点解码，看分子怎样一步步"变形"。

In [ ]:
idx_a, idx_b = 0, 7                                 # ✏️ 换两个分子试试
z_a, z_b = mu_test[idx_a], mu_test[idx_b]

steps = torch.linspace(0, 1, 8).unsqueeze(1)
z_path = z_a + (z_b - z_a) * steps                  # 直线插值
path_smiles = decode_latent(z_path)

path_mols, legends = [], []
for t, s in zip(steps.flatten().tolist(), path_smiles):
    c = canonical(s)
    if c:
        path_mols.append(Chem.MolFromSmiles(c))
        legends.append(f"t = {t:.2f}")
Draw.MolsToGridImage(path_mols, molsPerRow=4, subImgSize=(250, 190), legends=legends)

🤔 教科书上说 VAE 的插值是"平滑过渡"。这里是吗？

📘 **不是。** 两端（$t=0$ 和 $t=1$）是对两个真实 CEP 分子的重构，还算像稠环共轭体系；但中间几步跳得很厉害，出现了大环、链状分子，和两端都不像。

为什么？这要用一点**高维空间的几何**：

In [ ]:
norms = z_path.norm(dim=1)
for t, n in zip(steps.flatten().tolist(), norms.tolist()):
    print(f"t = {t:.2f}:  ||z|| = {n:.2f}")
print(f"\n真实分子的 ||μ|| 中位数: {mu_test.norm(dim=1).median():.2f}")

📘 看出来了吗：**两端的 $\|z\|$ 正常，中间明显变小**。

原因是高维空间里两个随机向量几乎总是接近**正交**的。若 $\|z_A\| = \|z_B\| = r$ 且两者正交，中点的长度是

$$\left\| \frac{z_A + z_B}{2} \right\| = \frac{r}{\sqrt 2} \approx 0.71\, r$$

也就是说，**直线插值的中点会掉进球壳内部的空腔里**——那里同样没有训练数据（第 6 节会看到，真实分子集中在一层球壳上）。解码器在那里没受过训练，于是输出奇怪的分子。

💡 **解决办法**：用**球面插值**（slerp）代替直线插值，让路径保持在球壳上。这在图像生成里是标准做法，练习 09a-3 会让你试试。

🤔 如果你有一个性能好的分子 A 和一个容易合成的分子 B，想沿着 A→B 采样一批"折中"的候选分子——根据上面的结果，直接用直线插值行吗？

### 5.2 潜空间长什么样？

▶️ 用 03a 的 PCA 把 128 维潜空间压到 2 维，按 PCE 上色：

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
coords = pca.fit_transform(mu_test.numpy())
pce_test = df["PCE"].values[test_idx]

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
sc = axes[0].scatter(coords[:, 0], coords[:, 1], c=pce_test, cmap="viridis", s=4)
plt.colorbar(sc, ax=axes[0], label="PCE (%)")
axes[0].set_title("VAE latent space (PCA), coloured by PCE")

heavy = np.array([Chem.MolFromSmiles(s).GetNumHeavyAtoms() for s in df["smiles"].values[test_idx]])
sc2 = axes[1].scatter(coords[:, 0], coords[:, 1], c=heavy, cmap="plasma", s=4)
plt.colorbar(sc2, ax=axes[1], label="heavy atoms")
axes[1].set_title("same map, coloured by molecule size")
for ax in axes:
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()
print("前两个主成分解释的方差:", pca.explained_variance_ratio_.round(3))

📘 分子大小（右图）在潜空间里是有结构的：大分子和小分子分别聚在不同区域。PCE（左图）就混乱得多。

🤔 为什么 PCE 看起来没什么规律？

📘 两个原因：

1. **VAE 训练时完全没看过 PCE**，它只被要求重建结构。潜空间按"结构相似"组织，而结构相似的分子 PCE 可以差很多（03b 第 4 节的教训）；
2. 128 维压到 2 维只保留了很小一部分方差（看上面打印的数字），很多信息在投影中丢掉了。

💡 要让潜空间"知道"性质，可以在训练时**同时**用一个小网络从 $z$ 预测 PCE（联合训练）。这正是 Gómez-Bombarelli 那篇论文的做法：这样就能在潜空间里沿着"PCE 变大"的方向做梯度上升。练习 09a-3 会让你试试这个思路的简化版。

---
## 6. 从先验采样：有效 ≠ 合理

📘 VAE 的承诺是：训练后潜空间接近 $N(0, I)$，所以可以**直接从标准正态分布里抽一个 $z$**，解码得到新分子。

▶️ 抽 1000 个试试：

In [ ]:
z_prior = torch.randn(1000, LATENT_DIM)
prior_smiles = decode_latent(z_prior)
prior_canonical = [canonical(s) for s in prior_smiles]
prior_valid = [c for c in prior_canonical if c]

canonical_train = {Chem.CanonSmiles(s) for s in df["smiles"]}
print(f"有效率:   {len(prior_valid) / 1000:.3f}")
print(f"唯一性:   {len(set(prior_valid)) / max(len(prior_valid), 1):.3f}")
print(f"新颖性:   {len(set(prior_valid) - canonical_train) / max(len(set(prior_valid)), 1):.3f}")

📘 有效率 90%、唯一性和新颖性接近 100%——和 08a 的 LSTM 在同一水平。而且别忘了：09a 第 1 节证明过，SELFIES 连随机乱改都 100% 合法，所以这个 valid 几乎是**白送**的。

🤔 真的吗？把分子画出来看看。

In [ ]:
Draw.MolsToGridImage([Chem.MolFromSmiles(c) for c in prior_valid[:12]], molsPerRow=4, subImgSize=(260, 200))

📘 **这些分子一点也不像 CEP 的分子。** 三元环、四元环、累积双键、一堆 sp³ 碳……它们在化学上"能存在"（至少化合价是对的），但和训练数据那些稠合芳环系统完全是两回事，作为有机光伏材料更是毫无道理。

▶️ 定量地比一比：

In [ ]:
train_sample = [Chem.MolFromSmiles(s) for s in df["smiles"].sample(2000, random_state=0)]
prior_mols = [Chem.MolFromSmiles(c) for c in prior_valid]

comparison = pd.DataFrame({
    "CEP training": [np.mean([m.GetNumHeavyAtoms() for m in train_sample]),
                     np.mean([Descriptors.NumAromaticRings(m) for m in train_sample]),
                     np.mean([Descriptors.FractionCSP3(m) for m in train_sample])],
    "VAE prior samples": [np.mean([m.GetNumHeavyAtoms() for m in prior_mols]),
                          np.mean([Descriptors.NumAromaticRings(m) for m in prior_mols]),
                          np.mean([Descriptors.FractionCSP3(m) for m in prior_mols])],
}, index=["heavy atoms", "aromatic rings", "fraction sp3 C"]).round(2)
comparison

📘 差距大得惊人，尤其是**芳香环**：训练数据平均每个分子有近 6 个芳香环，先验采样出来的平均**连 1 个都不到**。sp³ 碳的比例从 0.02 涨到 0.24，分子也小了四分之一。

对比一下：08a 的 LSTM 在同样的比较里几乎和训练数据完全重合。**一个 valid/unique/novel 三项全优的模型，生成的东西可以和训练数据毫无关系。**

### 6.1 为什么？"先验空洞"

📘 这是 VAE 的一个经典问题，叫 **prior hole（先验空洞）**：

- KL 项只是**鼓励**每个分子的后验接近 $N(0,I)$，并没有保证所有后验**加起来**等于 $N(0,I)$；
- 实际训练出来，编码器把分子塞进了标准正态分布里的一小团区域；
- 从 $N(0,I)$ 采样，大概率落在**编码器从来没用过的空白地带**，解码器在那里没有受过训练，只好胡乱输出。

▶️ 量一量就看清楚了：

In [ ]:
print(f"编码出来的 μ，每一维的标准差（平均）: {mu_test.std(0).mean():.3f}   先验应该是 1.000")
print(f"编码出来的 μ 到原点距离（中位数）: {mu_test.norm(dim=1).median():.2f}")
print(f"先验样本 z 到原点距离（中位数）:   {torch.randn(3000, LATENT_DIM).norm(dim=1).median():.2f}")

📘 每一维的标准差只有约 0.55（先验要求 1.0）；真实分子到原点的距离集中在 8.5 左右，而先验样本在 11.3 左右。**我们采样的那层球壳上，几乎没有真实分子。**

再回头看第 5.1 节：直线插值的中点跑到了 $\|z\| \approx 6$ 的内层，同样是空的。**球壳内外都是空的，真实分子只住在中间薄薄的一层。**

💡 高维空间的几何很反直觉：$d$ 维标准正态分布的样本几乎全部落在半径 $\sqrt{d}$ 附近的一层薄球壳上（这里 $\sqrt{128} \approx 11.3$），而不是聚在原点附近。所以"编码器只用了中间一小块"这件事，在高维下意味着先验样本几乎**全部**落空。

### 6.2 在真实分子附近采样

📘 既然问题是"采样采到了空地上"，那就别从先验采，改成**在某个真实分子的 $\mu$ 附近加一点噪声**：

In [ ]:
rows = []
base = mu_test[:300]
originals_300 = [canonical(s) for s in df["smiles"].values[test_idx[:300]]]
for sigma in [0.0, 0.3, 1.0, 3.0]:                        # ✏️
    z = base + sigma * torch.randn_like(base)
    out = [canonical(s) for s in decode_latent(z)]
    ok = [c for c in out if c]
    mols_ok = [Chem.MolFromSmiles(c) for c in ok]
    rows.append({
        "sigma": sigma,
        "valid": len(ok) / len(base),
        "same as source": np.mean([a == b for a, b in zip(out, originals_300)]),
        "aromatic rings": round(float(np.mean([Descriptors.NumAromaticRings(m) for m in mols_ok])), 2),
        "heavy atoms": round(float(np.mean([m.GetNumHeavyAtoms() for m in mols_ok])), 1),
    })
pd.DataFrame(rows).set_index("sigma")

📘 $\sigma = 0.3$ 时最有意思：只有 7% 和原分子相同（也就是说 90% 以上是**新分子**），但芳香环数（约 4）和分子大小（约 26 个重原子）仍然和训练数据接近——比先验采样的 0.41 个芳香环好太多了。$\sigma$ 再大就开始"跑远"，到 $\sigma = 3$ 基本和先验采样一样糟。

**这是实践中的一个重要技巧**：用 VAE 做分子设计时，与其从先验瞎抽，不如从已知的好分子出发，在它附近搜索。这也正是 09b 要用的思路。

### 🏋️ 练习 09a-1

$\beta$ 控制重构和规整之间的平衡。用 $\beta = 0.5$（比现在大 10 倍）重新训练 20 轮（其余不变），比较：

1. 训练结束时的重构损失和 KL；
2. 测试集完全还原率；
3. 编码出来的 $\mu$ 的标准差（第 6.1 节那个数）。

$\beta$ 变大以后，潜空间更接近先验了吗？重构变差了多少？这就是所谓的"重构-规整权衡"。

### 🏋️ 练习 09a-2

从训练集里随机抽 1000 个分子，当作一个"完美生成模型"的输出，算出第 6 节表格里的三列（重原子数、芳香环数、sp³ 碳比例），和 VAE 的先验采样放在一起比较。

再想一想：如果只看 valid / unique / novel 三个指标，这个"完美生成模型"和 VAE 哪个分数更高？这说明了什么？

### 🏋️ 练习 09a-3

两小题。

**(a) 球面插值。** 第 5.1 节说直线插值的中点会掉进空腔。改用 **slerp**（球面线性插值）重做一次：

```python
def slerp(a, b, t):
    a_n, b_n = a / a.norm(), b / b.norm()
    omega = torch.acos((a_n * b_n).sum().clamp(-1, 1))
    r = a.norm() + (b.norm() - a.norm()) * t          # 长度也线性过渡
    return r * (torch.sin((1 - t) * omega) * a_n + torch.sin(t * omega) * b_n) / torch.sin(omega)
```

打印路径上每一步的 $\|z\|$，确认它不再塌陷，再把分子画出来。中间的分子是不是更像两端了？

**(b) 在潜空间里做优化。**

1. 用 `mu_test`（128 维）和对应的 `pce_test` 训练一个岭回归（`sklearn.linear_model.Ridge`），用 `train_test_split` 看测试 R²——潜向量能预测 PCE 吗？
2. 取 PCE 最高的那个测试分子的 $\mu$，沿岭回归系数方向走几步：`z + step * coef / np.linalg.norm(coef)`，`step` 取 0 到 5；
3. 每一步解码并画出分子，同时打印岭回归的预测值。

预测的 PCE 一路上升了吗？分子还像有机光伏材料吗？这个练习会让你直观感受到 09b 要讲的"模型欺骗"问题。

In [ ]:
# 在这里写你的代码

---
## 7. RNN 还是 VAE？

| | SMILES LSTM（08a） | SELFIES VAE（09a） |
|---|---|---|
| 有效率 | 0.92–0.97（T=1.0），降温到 0.5 接近 1.00 | 约 0.90 |
| 分子像训练数据吗 | **很像** | **不像**（先验空洞） |
| 有没有分子的向量表示 | 没有 | **有**，可以插值、可以优化 |
| 训练难度 | 容易 | 要调 $\beta$，容易后验坍缩 |
| 典型用途 | 大量采样、微调做定向生成 | 潜空间优化、分子插值 |

📘 结论不是"谁更好"，而是**它们擅长的事不一样**，而且两者都有同一个教训：

> **不要只看 valid / unique / novel 这三个数字。** 它们能被轻松刷高（SELFIES 直接把 valid 拉满），却说明不了生成的分子是否有化学意义。一定要**把分子画出来看**，并和训练数据的性质分布比一比。

💡 现在主流的分子生成模型已经发展到：Transformer 语言模型（和大语言模型同源）、图生成模型（直接生成分子图，不经过字符串）、扩散模型（可以直接生成 3D 构象）。但这一节的三个核心概念——潜空间、重构-规整权衡、先验空洞——在那些模型里依然成立。

**下一节 09b**：到现在为止，生成的分子只是"像训练数据"。如果我们想要 **PCE 高**的分子呢？